In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

In [ ]:
class MakeDataset(Dataset):
    def __init__(self, x_data, labels=None):

        if isinstance(x_data, torch.Tensor):
            self.X = x_data.float()
        elif isinstance(x_data, np.ndarray):
            self.X = torch.from_numpy(x_data).float()
        else:
            raise TypeError(f"x_data must be a numpy array or torch.Tensor, got {type(x_data)}")

        if labels is not None:
            if isinstance(labels, torch.Tensor):
                y = labels.float()
            elif isinstance(labels, np.ndarray):
                y = torch.from_numpy(labels).float()
            else:
                raise TypeError(f"labels must be a numpy array or torch.Tensor, got {type(labels)}")

            if y.dim() == 1:
                y = y.unsqueeze(1)

            if y.size(0) != self.X.size(0):
                raise ValueError("inputs and labels must have the same length")

            self.y = y
        else:
            self.y = None

    def __len__(self):
        return self.X.size(0)

    def __getitem__(self, idx):
        x = self.X[idx]
        if self.y is None:
            return x
        return x, self.y[idx]

data_0 = np.load(f"/gpfs/bwfor/work/ws/hd_gy283-my_data/test_batch/batch_steps=1e6_0.npz")
data_1 = np.load(f"/gpfs/bwfor/work/ws/hd_gy283-my_data/test_batch/batch_steps=1e6_1.npz")
data_2 = np.load(f"/gpfs/bwfor/work/ws/hd_gy283-my_data/test_batch/batch_steps=1e6_2.npz")
data_3 = np.load(f"/gpfs/bwfor/work/ws/hd_gy283-my_data/test_batch/batch_steps=1e6_3.npz")
data_4 = np.load(f"/gpfs/bwfor/work/ws/hd_gy283-my_data/test_batch/batch_steps=1e6_4.npz")

inputs_0 = data_0["inputs"]
inputs_1 = data_1["inputs"]
inputs_2 = data_2["inputs"]
inputs_3 = data_3["inputs"]
inputs_4 = data_4["inputs"]

outputs_0 = data_0["currents"]
outputs_1 = data_1["currents"]
outputs_2 = data_2["currents"]
outputs_3 = data_3["currents"]
outputs_4 = data_4["currents"]


raw_inputs = np.concatenate([inputs_0, inputs_1, inputs_2, inputs_3, inputs_4])
outputs = np.concatenate([outputs_0, outputs_1, outputs_2, outputs_3, outputs_4])

inputs = raw_inputs[:, 1:]

print(f"input_shape = {inputs.shape}")
print(f"output_shape = {outputs.shape}")

X_train, X_test, y_train, y_test = train_test_split(inputs, outputs, test_size=0.2, random_state=42, shuffle=True)

eps = 1e-8
X_train_mean = X_train.mean(0)
X_train_std = X_train.std(0) + eps

y_train_mean = y_train.mean(0)
y_train_std = y_train.std(0) + eps

X_train_norm = (X_train - X_train_mean) / X_train_std
X_test_norm = (X_test - X_train_mean) / X_train_std

y_train_norm = (y_train - y_train_mean) / y_train_std
y_test_norm = (y_test - y_train_mean) / y_train_std

train_set = MakeDataset(X_train_norm, y_train_norm)
test_set = MakeDataset(X_test_norm, y_test_norm)

[ 1.41126395  0.07329047  1.02029943 ...  0.30362452 -0.35390548
 -0.1903334 ]
input_shape = (5000, 7)
output_shape = (5000,)


In [3]:
class NeuralNet(nn.Module):
    
    def __init__(
            self,
            in_features: int,
            out_features: int,
            hidden_dim: int,
            num_layers: int,
            dropout_p: float = 0.2
    ):
        super(NeuralNet, self).__init__()

        self.in_features = in_features
        self.out_features = out_features
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.dropout_p    = dropout_p

        self.model_layers = self.build_model()
        self.model = nn.Sequential(*self.model_layers)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        
        out = self.model(x)

        return out
    
    def build_model(self):
        layer_list = []

        for l in range(self.num_layers):
            # Determine dims for this layer
            if l == 0:
                in_dim, out_dim = self.in_features, self.hidden_dim
            elif l == self.num_layers - 1:
                in_dim, out_dim = self.hidden_dim, self.out_features
            else:
                in_dim = out_dim = self.hidden_dim

            # Linear
            layer_list.append(nn.Linear(in_dim, out_dim))

            # If not final layer, add BN, activation, dropout
            if l < self.num_layers - 1:
                layer_list.append(nn.BatchNorm1d(out_dim))
                layer_list.append(nn.SiLU())
                layer_list.append(nn.Dropout(self.dropout_p))

        return layer_list

In [ ]:
X_mean = torch.from_numpy(X_train_mean).float()
X_std  = torch.from_numpy(X_train_std).float()
y_mean = torch.tensor(y_train_mean).float()
y_std  = torch.tensor(y_train_std).float()

in_features = 8
out_features = 1
hidden_dim = 64
num_layers = 4

model = NeuralNet(in_features=in_features, out_features=out_features, hidden_dim=hidden_dim, num_layers=num_layers)

state_dict = torch.load(f="../../SM_0.pth", map_location=torch.device("cpu"))

model.load_state_dict(state_dict=state_dict)
model.eval()

sim_curve = np.load("/gpfs/bwfor/work/ws/hd_gy283-my_data/test_folder/acceptors_seed=543763.npz")
print(sim_curve.files)
print(sim_curve["current"].shape)
curve = sim_curve["current"]
control_volts = sim_curve["control"]

vs_ = torch.linspace(-1.5, 1.5, 100)

n = 100
vs = torch.linspace(-1.5, 1.5, n)

control_volts[0] = 0.0
control_volts[1] = 0.0

x = torch.from_numpy(control_volts).float().unsqueeze(0).repeat(n, 1)
x[:, 1] = vs

x_norm = (x - X_mean) / X_std

with torch.no_grad():
    out = model(x_norm) * y_std + y_mean

plot_kwargs = {
    "lw": 3.5,
    "alpha": 0.75
}
plt.scatter(vs_, curve)
plt.plot(vs, out.detach(), **plot_kwargs, zorder=3)
ax = plt.gca()
ax.grid(True)
ax.set_facecolor("lightgrey")
ax.xaxis.grid(color="w", lw=1.)
ax.yaxis.grid(color="w", lw=1.)
ax.xaxis.set_tick_params(width=2)
ax.yaxis.set_tick_params(width=2)
ax.spines[['top', 'bottom', 'right', 'left']].set_linewidth(2)

odict_items([('model.0.weight', tensor([[-5.7519e-41,  1.5457e-01,  6.8886e-02, -1.9586e-02,  1.0330e-01,
          1.2385e-01,  7.7205e-02,  9.3199e-02],
        [ 5.0150e-41, -2.1921e-01, -1.8913e-01,  1.0987e-02,  2.2964e-01,
          8.1489e-02, -6.1491e-03, -5.0769e-02],
        [-4.9457e-41,  9.5011e-02,  6.3866e-02,  3.7576e-02,  1.2501e-01,
          1.2890e-01,  1.2761e-01, -4.3873e-02],
        [ 4.9659e-41,  2.4049e-01,  5.6144e-02,  1.9564e-02, -2.2330e-02,
         -1.3336e-01, -2.1102e-01,  4.0807e-02],
        [ 5.0920e-41,  3.7287e-01,  4.0797e-02,  2.2857e-02,  2.0179e-02,
          5.9815e-03, -2.2034e-02, -6.3947e-02],
        [-6.2519e-41,  1.0737e-01,  1.9921e-03, -1.9042e-01,  6.2831e-02,
          5.8028e-02,  1.3477e-01, -3.2163e-02],
        [-4.9577e-41, -1.0591e-01, -6.1235e-02,  8.6828e-02, -1.0269e-02,
          1.4199e-01,  1.3731e-01,  4.6302e-02],
        [ 4.9201e-41, -8.1295e-03,  9.2669e-03, -4.6288e-03,  1.3510e-02,
         -3.3083e-02, -1.3038e-01

RuntimeError: Error(s) in loading state_dict for NeuralNet:
	Missing key(s) in state_dict: "model.1.weight", "model.1.bias", "model.1.running_mean", "model.1.running_var", "model.5.weight", "model.5.bias", "model.5.running_mean", "model.5.running_var", "model.9.weight", "model.9.bias", "model.9.running_mean", "model.9.running_var", "model.12.weight", "model.12.bias". 
	Unexpected key(s) in state_dict: "model.2.weight", "model.2.bias", "model.6.weight", "model.6.bias". 
	size mismatch for model.8.weight: copying a param with shape torch.Size([1, 64]) from checkpoint, the shape in current model is torch.Size([64, 64]).
	size mismatch for model.8.bias: copying a param with shape torch.Size([1]) from checkpoint, the shape in current model is torch.Size([64]).